# Density model for electron density and NS distribution

This notebook creates a numpy 2d array that stores the electron density distribution in polar coordinates $(r, \phi)$ used to sample the neutron star positions in the Galaxy. The array can be saved as a .npy object.

In [ ]:
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pygedm
import scipy as sc
import scipy.integrate as integrate

from mlpoppyns.simulator.config_simulator import cfg
import utilities.samplers.random_sampler as rs
import utilities.plot_settings

In [ ]:
# Create empty arrays for NE2001 and YMW16 model output
n_bins = 500
r_edges = np.linspace(0.0, cfg["r_extent"], n_bins + 1)
phi_edges = np.linspace(0.0, 2.0 * np.pi, n_bins + 1)

x_grid = np.zeros((len(r_edges), len(phi_edges)))
y_grid = np.zeros((len(r_edges), len(phi_edges)))

d_ymw16 = np.zeros((len(r_edges), len(phi_edges)))
d_ne2001 = np.zeros((len(r_edges), len(phi_edges)))

# Loop through coordinates and compute the value of the electron density.
for i in range(len(r_edges)):
    for j in range(len(phi_edges)):
        # generate x, y, z in pc from the grid edges in polar coordinates defined above.
        x_grid[i, j] = r_edges[i] * 1.0e3 * np.cos(phi_edges[j])
        y_grid[i, j] = r_edges[i] * 1.0e3 * np.sin(phi_edges[j])
        z = 0.0

        # Compute electron density at each point.
        d_ymw16[i, j] = pygedm.calculate_electron_density_xyz(
            x_grid[i, j], y_grid[i, j], z, method="ymw16"
        ).value
        d_ne2001[i, j] = pygedm.calculate_electron_density_xyz(
            x_grid[i, j], y_grid[i, j], z, method="ne2001"
        ).value

        # Correct the density for the element of area.
        d_ymw16[i, j] = d_ymw16[i, j] * r_edges[i]
        d_ne2001[i, j] = d_ne2001[i, j] * r_edges[i]

# Normalize the densities.
d_ymw16 = d_ymw16 / np.max(d_ymw16)
d_ne2001 = d_ne2001 / np.max(d_ne2001)

"""
# Save to a .npy file.
with open('YMW16_density_model.npy', 'wb') as f:
    np.save(f, d_ymw16)
"""

In [ ]:
# Plot the electron density in polar coordinates.
plt.figure(figsize=(4 * 3 * 2, 3 * 3))
plt.subplot(1, 2, 1)
plt.title("NE2001 electron density model, z=0 plane")
plt.imshow(np.log10(d_ne2001).T, extent=(0, 20, 0, 2.0 * np.pi), clim=(-8, 0))
plt.xlabel("r [kpc]")
plt.ylabel("phi [rad]")
cbar = plt.colorbar()
cbar.set_label("log10($N_e$)")

plt.subplot(1, 2, 2)
plt.title("YMW16 electron density model, z=0 plane")
plt.imshow(np.log10(d_ymw16).T, extent=(0, 20, 0, 2.0 * np.pi), clim=(-8, 0))
plt.xlabel("r [kpc]")
plt.ylabel("phi [rad]")
cbar = plt.colorbar()
cbar.set_label("log10($N_e$)")

# plt.savefig("plot_model_ne.png")
plt.show()

In [ ]:
# Plot the grid resolution.
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"x [kpc]")
ax.set_ylabel(r"y [kpc]")

ax.plot(
    x_grid,
    y_grid,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=1,
    rasterized=True,
)


plt.show()

In [ ]:
# Sample random points from the electron density distributions.
r_rand_ne2001, phi_rand_ne2001 = rs.random_from_pdf_2d(
    r_edges, phi_edges, d_ne2001, 100000
)
x_rand_ne2001 = r_rand_ne2001 * np.cos(phi_rand_ne2001)
y_rand_ne2001 = r_rand_ne2001 * np.sin(phi_rand_ne2001)

r_rand_ymw16, phi_rand_ymw16 = rs.random_from_pdf_2d(
    r_edges, phi_edges, d_ymw16, 100000
)
x_rand_ymw16 = r_rand_ymw16 * np.cos(phi_rand_ymw16)
y_rand_ymw16 = r_rand_ymw16 * np.sin(phi_rand_ymw16)

In [ ]:
# Plot the distribution from the ne2001 model.
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"x [kpc]")
ax.set_ylabel(r"y [kpc]")

ax.plot(
    x_rand_ne2001,
    y_rand_ne2001,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)


plt.show()

In [ ]:
# Plot the distribution from the ymw16 model.
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"x [kpc]")
ax.set_ylabel(r"y [kpc]")

ax.plot(
    x_rand_ymw16,
    y_rand_ymw16,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)


plt.show()